# [16.2] KernelSHAP and PartitionSHAP Controls

This notebook continues the Shapley baseline track. You will solve full-table
KernelSHAP as constrained weighted regression, then implement PartitionSHAP as
exact Owen values for a declared partition.

<img src="../../instructions/assets/kernelshap_partition_validation_loop.svg" width="760">

The core question is narrow: when can an approximate SHAP method be trusted on
a finite model organism? First make the coalition table complete and the oracle
known. Only then read the CUDA report for a trained neural coalition game.

<details><summary>Help - why start from complete tables?</summary>

If the full coalition table is known, then mistakes in kernel weights,
efficiency constraints, grouping assumptions, and interaction handling are not
hidden behind sampling noise. This notebook is a calibration point for later
sampled SHAP and token attribution sections.

</details>


## Setup

Run the setup cell once. The public tests use tiny exact games; they are meant
to catch conceptual errors before the final CUDA report is inspected.

<details><summary>Expected output</summary>

No printed output. Imports should succeed and the dataclasses should be defined.

</details>


In [1]:
from collections.abc import Callable, Mapping
from dataclasses import dataclass
import itertools
import json
import math
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part2_kernelshap_partition_shap_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_kernelshap_partition_shap_controls.tests as tests

Coalition = frozenset[int]


@dataclass(frozen=True)
class KernelSHAPApproximationReport:
    shapley_values: t.Tensor
    exact_values: t.Tensor
    max_abs_error: float
    approximates_exact: bool


@dataclass(frozen=True)
class PartitionSHAPReport:
    group_values: t.Tensor
    player_values: t.Tensor
    exact_values: t.Tensor
    max_abs_error: float
    recovers_exact: bool


## Exact-game helpers

These helpers are inherited from 16.1 so this notebook can focus on the new
KernelSHAP and PartitionSHAP logic.

<details><summary>Expected output</summary>

No printed output. You should have `all_coalitions`, `exact_shapley_values`,
`additive_game`, and `conjunction_game` available.

</details>


In [2]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    if num_players <= 0:
        raise ValueError("num_players must be positive.")
    players = range(num_players)
    coalitions = []
    for size in range(num_players + 1):
        coalitions.extend(frozenset(group) for group in itertools.combinations(players, size))
    return tuple(coalitions)


def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    values = {frozenset(key): float(value) for key, value in coalition_values.items()}
    expected = set(all_coalitions(num_players))
    missing = expected - set(values)
    if missing:
        raise ValueError(f"coalition value table is missing {len(missing)} coalitions.")
    return values


def coalition_values_from_function(
    num_players: int,
    value_fn: Callable[[Coalition], float],
) -> dict[Coalition, float]:
    return {coalition: float(value_fn(coalition)) for coalition in all_coalitions(num_players)}


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    values = normalize_coalition_values(coalition_values, num_players=num_players)
    denominator = math.factorial(num_players)
    shapley = t.zeros(num_players, dtype=t.float64)
    for player in range(num_players):
        others = [item for item in range(num_players) if item != player]
        for size in range(num_players):
            weight = (
                math.factorial(size)
                * math.factorial(num_players - size - 1)
                / denominator
            )
            for group in itertools.combinations(others, size):
                coalition = frozenset(group)
                shapley[player] += weight * (
                    values[coalition | {player}] - values[coalition]
                )
    return shapley


def additive_game(weights: t.Tensor) -> dict[Coalition, float]:
    weights = weights.flatten().double()
    return coalition_values_from_function(
        int(weights.numel()),
        lambda coalition: weights[list(coalition)].sum().item() if coalition else 0.0,
    )


def conjunction_game(num_players: int) -> dict[Coalition, float]:
    full = frozenset(range(num_players))
    return coalition_values_from_function(num_players, lambda coalition: coalition == full)


## Exercise 1 - implement the finite KernelSHAP weight

For a coalition of size `s` among `n` players, return:

```text
(n - 1) / (binom(n, s) * s * (n - s))
```

Reject empty and full coalitions; they belong to the baseline and efficiency
constraint, not to the finite-weight rows.

<details><summary>Help - why reject empty and full coalitions?</summary>

The empty coalition anchors the baseline. The full coalition anchors the total
value delta. Including either one as an ordinary finite row usually hides a
broken equality constraint.

</details>

<details><summary>Common bugs</summary>

- Forgetting the combinatorial `math.comb` term.
- Returning a weight for the empty coalition.
- Using uniform weights, which can pass easy additive cases by accident.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_kernelshap_kernel_weight_uses_finite_coalition_formula` passed!
```

</details>

<details><summary>Solution</summary>

Check `0 < coalition_size < num_players`, then implement the formula directly
in `float64`-compatible Python arithmetic.

</details>


In [3]:
def kernelshap_kernel_weight(coalition_size: int, num_players: int) -> float:
    if coalition_size <= 0 or coalition_size >= num_players:
        raise ValueError("KernelSHAP finite weights require 0 < coalition_size < num_players.")
    return (num_players - 1) / (
        math.comb(num_players, coalition_size)
        * coalition_size
        * (num_players - coalition_size)
    )


tests.test_kernelshap_kernel_weight_uses_finite_coalition_formula(
    kernelshap_kernel_weight,
)


All tests in `test_kernelshap_kernel_weight_uses_finite_coalition_formula` passed!


## Exercise 2 - solve full-table KernelSHAP

Build the non-empty, non-full coalition mask matrix, weight each row by the
KernelSHAP kernel, and solve weighted least squares with the hard constraint
`sum_i phi_i = v(full) - v(empty)`.

<details><summary>Help - what does the constrained solve look like?</summary>

Use the normal equations `X.T @ W @ X` and add one KKT row/column for the
all-ones equality constraint. The right-hand side gets one extra entry:
`v(full) - v(empty)`.

</details>

<details><summary>Common bugs</summary>

- Solving unconstrained least squares and hoping efficiency works out.
- Forgetting to subtract `v(empty)` from intermediate coalition values.
- Including the full coalition as a finite row.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_kernelshap_approximation_report_matches_exact_additive_game` passed!
```

</details>

<details><summary>Solution</summary>

Accumulate design rows, targets, and weights, solve the KKT system, then compare
against `exact_shapley_values` in a report object.

</details>


In [4]:
def kernelshap_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    if num_players < 2:
        raise ValueError("KernelSHAP needs at least two players for finite weights.")
    values = normalize_coalition_values(coalition_values, num_players=num_players)
    empty = frozenset()
    full = frozenset(range(num_players))
    baseline = values[empty]
    total_delta = values[full] - baseline

    rows = []
    targets = []
    weights = []
    for coalition in all_coalitions(num_players):
        size = len(coalition)
        if size == 0 or size == num_players:
            continue
        rows.append([1.0 if player in coalition else 0.0 for player in range(num_players)])
        targets.append(values[coalition] - baseline)
        weights.append(kernelshap_kernel_weight(size, num_players))

    design = t.tensor(rows, dtype=t.float64)
    target = t.tensor(targets, dtype=t.float64)
    weight_vector = t.tensor(weights, dtype=t.float64)
    hessian = design.T @ (design * weight_vector.unsqueeze(-1))
    rhs = design.T @ (target * weight_vector)
    constraint = t.ones((1, num_players), dtype=t.float64)
    kkt = t.cat(
        [
            t.cat([hessian, constraint.T], dim=1),
            t.cat([constraint, t.zeros((1, 1), dtype=t.float64)], dim=1),
        ],
        dim=0,
    )
    constrained_rhs = t.cat([rhs, t.tensor([total_delta], dtype=t.float64)])
    return t.linalg.solve(kkt, constrained_rhs)[:num_players]


def kernelshap_approximation_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> KernelSHAPApproximationReport:
    kernel_values = kernelshap_values(coalition_values, num_players=num_players)
    exact = exact_shapley_values(coalition_values, num_players=num_players)
    max_error = float((kernel_values - exact).abs().max().item())
    return KernelSHAPApproximationReport(
        shapley_values=kernel_values,
        exact_values=exact,
        max_abs_error=max_error,
        approximates_exact=max_error <= tolerance,
    )


tests.test_kernelshap_approximation_report_matches_exact_additive_game(
    kernelshap_approximation_report,
    additive_game,
)


All tests in `test_kernelshap_approximation_report_matches_exact_additive_game` passed!


## Exercise 3 - split credit in an interaction game

A three-player conjunction has value only when all players are present.
Full-table KernelSHAP should split the value one third each, matching exact
Shapley.

<details><summary>Expected output</summary>

```text
All tests in `test_kernelshap_interaction_report_splits_conjunction_credit` passed!
```

</details>

<details><summary>Interpreting the result</summary>

This is where leave-one-out intuition starts to fail. Every player is necessary,
but exact Shapley credit is averaged across contexts rather than assigned from
one full-coalition deletion.

</details>


In [5]:
tests.test_kernelshap_interaction_report_splits_conjunction_credit(
    kernelshap_approximation_report,
    conjunction_game,
)


All tests in `test_kernelshap_interaction_report_splits_conjunction_credit` passed!


## Exercise 4 - collapse player coalitions into group coalitions

For `groups=((0, 1), (2, 3))`, group coalition `{1}` means players `{2, 3}`.
Expand group indices back to player indices before reading the value table.

<details><summary>Help - what should the additive example return?</summary>

For weights `[1, 2, 3, 4]`, singleton group values should be `3.0` and `7.0`,
because group 0 contains players 0 and 1, while group 1 contains players 2 and
3.

</details>

<details><summary>Common bugs</summary>

- Reading group index `1` as player `1`.
- Forgetting the empty group coalition.
- Returning player-coalition keys instead of group-coalition keys.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_grouped_coalition_values_expand_groups_before_lookup` passed!
```

</details>

<details><summary>Solution</summary>

Enumerate coalitions over group indices, union the players inside selected
groups, and read that player coalition from the original table.

</details>


In [6]:
def grouped_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    groups: tuple[tuple[int, ...], ...],
) -> dict[Coalition, float]:
    group_count = len(groups)
    player_count = sum(len(group) for group in groups)
    values = normalize_coalition_values(coalition_values, num_players=player_count)
    group_values = {}
    for group_coalition in all_coalitions(group_count):
        players: set[int] = set()
        for group_index in group_coalition:
            players.update(groups[group_index])
        group_values[group_coalition] = values[frozenset(players)]
    return group_values


tests.test_grouped_coalition_values_expand_groups_before_lookup(
    grouped_coalition_values,
    additive_game,
)


All tests in `test_grouped_coalition_values_expand_groups_before_lookup` passed!


## Exercise 5 - compute exact Owen values

For each player, average marginal contributions over two choices: which other
groups came before this player's group, and which other players inside this
player's group came before this player. This is the exact PartitionSHAP/Owen
calculation for a fixed partition.

<details><summary>Help - what loops are required?</summary>

For a player `i` in group `G`, enumerate subsets of other groups and subsets of
other players inside `G`. The marginal is measured by adding `i` to the union of
those outside and inside players.

</details>

<details><summary>Common bugs</summary>

- Allowing overlapping groups.
- Splitting group-level credit uniformly and calling it Owen values.
- Forgetting to compare the final player values to exact Shapley.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_partition_shap_report_recovers_additive_groups` passed!
All tests in `test_partition_shap_report_rejects_invalid_grouping` passed!
```

</details>

<details><summary>Solution</summary>

Use Shapley weights over outside group subsets and within-group player subsets,
then accumulate weighted marginal effects directly into each original player.

</details>


In [7]:
def partition_shap_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    groups: tuple[tuple[int, ...], ...],
) -> t.Tensor:
    if not groups:
        raise ValueError("groups must be nonempty.")
    flattened = [player for group in groups for player in group]
    if sorted(flattened) != list(range(len(flattened))):
        raise ValueError("groups must partition players 0..n-1 exactly once.")

    values = normalize_coalition_values(coalition_values, num_players=len(flattened))
    group_count = len(groups)
    player_values = t.zeros(len(flattened), dtype=t.float64)
    for group_index, group in enumerate(groups):
        outside_group_indices = [idx for idx in range(group_count) if idx != group_index]
        group_denominator = math.factorial(group_count)
        local_denominator = math.factorial(len(group))
        for local_index, player in enumerate(group):
            others_in_group = [
                candidate
                for idx, candidate in enumerate(group)
                if idx != local_index
            ]
            for outside_size in range(group_count):
                outside_weight = (
                    math.factorial(outside_size)
                    * math.factorial(group_count - outside_size - 1)
                    / group_denominator
                )
                for outside_groups in itertools.combinations(outside_group_indices, outside_size):
                    outside_players: set[int] = set()
                    for outside_group in outside_groups:
                        outside_players.update(groups[outside_group])
                    for inside_size in range(len(group)):
                        inside_weight = (
                            math.factorial(inside_size)
                            * math.factorial(len(group) - inside_size - 1)
                            / local_denominator
                        )
                        for inside_players in itertools.combinations(
                            others_in_group,
                            inside_size,
                        ):
                            coalition = frozenset(outside_players | set(inside_players))
                            player_values[player] += outside_weight * inside_weight * (
                                values[coalition | {player}] - values[coalition]
                            )
    return player_values


def partition_shap_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    groups: tuple[tuple[int, ...], ...],
    tolerance: float = 1e-9,
) -> PartitionSHAPReport:
    num_players = sum(len(group) for group in groups)
    player_values = partition_shap_values(coalition_values, groups=groups)
    exact = exact_shapley_values(coalition_values, num_players=num_players)
    group_values = exact_shapley_values(
        grouped_coalition_values(coalition_values, groups=groups),
        num_players=len(groups),
    )
    max_error = float((player_values - exact).abs().max().item())
    return PartitionSHAPReport(
        group_values=group_values,
        player_values=player_values,
        exact_values=exact,
        max_abs_error=max_error,
        recovers_exact=max_error <= tolerance,
    )


tests.test_partition_shap_report_recovers_additive_groups(
    partition_shap_report,
    additive_game,
)
tests.test_partition_shap_report_rejects_invalid_grouping(
    partition_shap_report,
    additive_game,
)


All tests in `test_partition_shap_report_recovers_additive_groups` passed!
All tests in `test_partition_shap_report_rejects_invalid_grouping` passed!


## Exercise 6 - split a pure interaction group symmetrically

When both players in a two-player conjunction are inside one group, the exact
Owen values should split credit equally.

<details><summary>Expected output</summary>

```text
All tests in `test_partition_shap_report_splits_interaction_group_symmetrically` passed!
```

</details>

<details><summary>Interpreting the result</summary>

This is the easy grouping case: the interacting players live in the same group,
so the within-group ordering is enough to split credit.

</details>


In [8]:
tests.test_partition_shap_report_splits_interaction_group_symmetrically(
    partition_shap_report,
    conjunction_game,
)


All tests in `test_partition_shap_report_splits_interaction_group_symmetrically` passed!


## Exercise 7 - anomaly hunt: reject the within-group shortcut

Use the game `v(S)=1` when players `0` and `2` are both present. With groups
`((0, 1), (2,))`, player `1` is in the same group as player `0` but is
irrelevant to the interaction. The correct player values are `[0.5, 0.0, 0.5]`.

<details><summary>Help - what failure are you trying to catch?</summary>

A shortcut that computes group credit and then splits the first group uniformly
will assign player `1` positive credit. That is a pretty-looking but wrong
result, because player `1` never changes the value of the game.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_partition_shap_report_handles_cross_group_interaction_without_irrelevant_credit` passed!
```

</details>


In [9]:
tests.test_partition_shap_report_handles_cross_group_interaction_without_irrelevant_credit(
    partition_shap_report,
    coalition_values_from_function,
)


All tests in `test_partition_shap_report_handles_cross_group_interaction_without_irrelevant_credit` passed!


## Exercise 8 - assemble the notebook contract and read the report

The smoke contract is fast and exact. The committed CUDA report is the slower
trained-model path: train the finite neural game, recover coalition values from
real model ablations, and reject the shuffled-label control.

<details><summary>Help - why not call the smoke test the GPU result?</summary>

The smoke test proves the learner-facing math on tiny finite games. The CUDA
report proves that the same controls pass on a trained neural coalition game.
They answer different questions and should stay separate.

</details>

<details><summary>Common bugs</summary>

- Returning tensors directly instead of JSON-ready lists.
- Omitting the cross-group anomaly from the smoke test.
- Claiming sampled large-model SHAP reliability from this finite report.

</details>

<details><summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
All tests in `test_committed_report_has_real_cuda_kernel_partition_evidence` passed!
```

</details>

<details><summary>Solution</summary>

Convert report tensors to lists at the smoke-test boundary. Read the committed
report and assert its CUDA evidence fields directly.

</details>


In [10]:
def _tensor_report(report: object) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def kernel_additive_smoke_test() -> dict:
    values = additive_game(t.tensor([1.0, -2.0, 0.5]))
    return _tensor_report(kernelshap_approximation_report(values, num_players=3))


def kernel_interaction_smoke_test() -> dict:
    values = conjunction_game(3)
    return _tensor_report(kernelshap_approximation_report(values, num_players=3))


def partition_additive_smoke_test() -> dict:
    values = additive_game(t.tensor([1.0, 2.0, 3.0, 4.0]))
    return _tensor_report(partition_shap_report(values, groups=((0, 1), (2, 3))))


def partition_interaction_smoke_test() -> dict:
    values = conjunction_game(2)
    return _tensor_report(partition_shap_report(values, groups=((0, 1),)))


def partition_cross_group_interaction_smoke_test() -> dict:
    values = coalition_values_from_function(
        3,
        lambda coalition: {0, 2}.issubset(coalition),
    )
    return _tensor_report(partition_shap_report(values, groups=((0, 1), (2,))))


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "kernel_additive": kernel_additive_smoke_test(),
        "kernel_interaction": kernel_interaction_smoke_test(),
        "partition_additive": partition_additive_smoke_test(),
        "partition_interaction": partition_interaction_smoke_test(),
        "partition_cross_group_interaction": partition_cross_group_interaction_smoke_test(),
    }


def load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    gpu = report["metrics"]["gpu_test"]
    assert report["accepted"] is True and report["tests_passed"] is True
    assert gpu["cuda_available"] is True and gpu["preflight_passed"] is True
    assert gpu["fit_mse"] <= 1e-8
    assert gpu["kernel_approximates_exact"] is True
    assert gpu["kernel_vs_true_max_abs_error"] <= 1e-4
    assert gpu["partition_recovers_exact"] is True
    assert gpu["aligned_partition_recovers_exact"] is True
    assert gpu["mismatched_partition_recovers_exact"] is True
    assert gpu["cross_group_partition_recovers_exact"] is True
    assert gpu["cross_group_irrelevant_credit"] <= 1e-8
    assert gpu["shuffled_control_rejected"] is True
    assert gpu["within_vram_budget"] is True
    return report


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    _ = max_vram_gb
    return load_committed_gpu_report()["metrics"]["gpu_test"]


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


tests.test_notebook_contract(run_smoke_test)
tests.test_committed_report_has_real_cuda_kernel_partition_evidence()

report = load_committed_gpu_report()
gpu = report["metrics"]["gpu_test"]
print("device:", gpu["device"])
print("torch:", gpu["torch_version"])
print("cuda:", gpu["cuda_version"])
print("kernel_vs_true_max_abs_error:", gpu["kernel_vs_true_max_abs_error"])
print("cross_group_irrelevant_credit:", gpu["cross_group_irrelevant_credit"])
print("shuffled_control_kernel_vs_true_error:", gpu["shuffled_control_kernel_vs_true_error"])
print("peak_vram_gb:", gpu["peak_vram_gb"])


All tests in `test_notebook_contract` passed!
All tests in `test_committed_report_has_real_cuda_kernel_partition_evidence` passed!
device: NVIDIA GeForce RTX 5090 Laptop GPU
torch: 2.12.1+cu132
cuda: 13.2
kernel_vs_true_max_abs_error: 9.375313920756412e-07
cross_group_irrelevant_credit: 0.0
shuffled_control_kernel_vs_true_error: 4.633333171407381
peak_vram_gb: 0.06262922286987305


## Signature Result

<img src="../../instructions/assets/kernelshap_partition_signature_result.svg" width="760">

| Check | Expected | Observed |
| --- | ---: | ---: |
| Complete coalition table | 16 | 16 |
| Neural fit MSE | <= 1e-8 | 1.102e-12 |
| KernelSHAP vs exact on model table | <= 1e-8 | 4.441e-16 |
| KernelSHAP vs analytic truth | <= 1e-4 | 9.375e-7 |
| Cross-group irrelevant credit | <= 1e-8 | 0.0 |
| Shuffled-label true error | >= 1.0 | 4.6333 |
| Peak VRAM | <= 1.0 GB | 0.063 GB |

<details><summary>Help - how should this result be interpreted?</summary>

This result says full-table KernelSHAP and exact Owen-value PartitionSHAP pass
on a complete finite table recovered from a trained CUDA model. It does not say
sampled SHAP is reliable on large language or vision models.

</details>


## Limitations

- This is a GT-0 complete-table model organism, not a sampled large-model SHAP result.
- The trained CUDA game has four binary features and 16 examples.
- The PartitionSHAP result depends on the declared partition.
- Exact Owen-value enumeration is exponential and is kept small here on purpose.

## Bonus anomaly hunting

- Implement the bad group-then-uniform-split shortcut and show that it assigns
  positive credit to player `1` in the cross-group interaction game.
- Change the grouping to singleton groups and confirm that PartitionSHAP reduces
  to ordinary exact Shapley.
- Sample only part of the coalition table and measure how quickly KernelSHAP
  stops matching the exact reference.
